In [1]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import numpy as np
import os

In [2]:
data_pipeline = "baseline"

In [3]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [4]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [5]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [6]:
X = pd.read_csv(Path(data_path) / "raw/train.csv")
X_test = pd.read_csv(Path(data_path) / "raw/test.csv")

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  str    
 11  stress_level             636221 non-null  str    
 12  academic_work_impact     647145 non-null  str    
 13  addicted_label           691369 non-null  int64  
dtypes: float64(9), 

In [7]:
y = X[target_column]
X = X.drop(target_column, axis=1)

In [8]:
cat_cols = ["gender"]
ordinal_cols = ["stress_level"]
binary_cols = ["academic_work_impact"]

In [9]:
for frame in [X, X_test]:
    frame.drop('id', axis=1, inplace=True)

    for col in cat_cols:
        frame[col] = frame[col].astype('category')

    frame['stress_level'] = frame['stress_level'].replace({'Low':0, 'Medium':1, 'High':2})
    frame['academic_work_impact'] = frame['academic_work_impact'].replace({'No':0, 'Yes':1})

    for col in ordinal_cols + binary_cols:
        frame[col] = frame[col].astype(np.float64)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   age                      662440 non-null  float64 
 1   daily_screen_time_hours  595515 non-null  float64 
 2   social_media_hours       557374 non-null  float64 
 3   gaming_hours             564548 non-null  float64 
 4   work_study_hours         639851 non-null  float64 
 5   sleep_hours              646889 non-null  float64 
 6   notifications_per_day    623785 non-null  float64 
 7   app_opens_per_day        610659 non-null  float64 
 8   weekend_screen_time      579306 non-null  float64 
 9   gender                   662335 non-null  category
 10  stress_level             636221 non-null  float64 
 11  academic_work_impact     647145 non-null  float64 
dtypes: category(1), float64(11)
memory usage: 58.7 MB


In [10]:
write(experiment_path / f"train.parq", X)
write(experiment_path / f"test.parq", X_test)